# Notebook 03 — Integração com LLM (Claude)

**Tech Challenge FIAP — Fase 2**

Este notebook demonstra a integração com o modelo Claude (Anthropic) para geração de explicações clínicas e relatórios narrativos sobre as predições do classificador de risco gestacional.

**Funcionalidades demonstradas:**
- Diagnóstico explicado em linguagem natural para o médico obstetra
- Modo Q&A — perguntas livres sobre o caso clínico
- Relatório narrativo comparando baseline vs. modelo otimizado pelo Algoritmo Genético

## 0. Imports e configurações

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from src.llm import interpret_diagnosis, generate_optimization_report, answer_question

print(f"Projeto raiz: {PROJECT_ROOT}")

Projeto raiz: /Users/igornatanael/workspace/tech-challenge-fiap-2


## 1. Carregar dados e modelo otimizado

In [2]:
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"

with open(EXPERIMENTS_DIR / "baseline_results.json", encoding="utf-8") as f:
    baseline_results = json.load(f)

print("Baseline carregado:")
print(json.dumps(baseline_results["random_forest"], indent=2))

experiment_01_path = EXPERIMENTS_DIR / "experiment_01_default.json"
with open(experiment_01_path, encoding="utf-8") as f:
    experiment_01 = json.load(f)

ag_status = experiment_01.get("status", "unknown")
print(f"\nStatus experimento AG 01: {ag_status}")

# Reconstruir pipeline do melhor modelo (RF baseline com hiperparâmetros do GridSearch)
from src.pipelines.preprocessing import build_preprocessed_splits
from src.models.baseline import build_pipelines
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test, scaler = build_preprocessed_splits(
    add_pulse_pressure=False,
    add_age_features=False,
    convert_temp=True,
    scale=True,
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_leaf=1,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=1,
    )),
])
rf_pipeline.fit(X_train, y_train)

# Feature importance
feature_names = X_train.columns.tolist()
importancias_raw = rf_pipeline.named_steps["model"].feature_importances_
feature_importance = dict(zip(feature_names, [round(float(v), 4) for v in importancias_raw]))
feature_importance = dict(sorted(feature_importance.items(), key=lambda x: x[1], reverse=True))

print(f"\nFeature importance: {feature_importance}")

Baseline carregado:
{
  "accuracy": 0.8987,
  "f1_macro": 0.9017,
  "recall_low_risk": 0.8841,
  "recall_mid_risk": 0.88,
  "recall_high_risk": 0.9487,
  "precision_macro": 0.8994,
  "roc_auc_macro_ovr": 0.9813,
  "best_params": {
    "model__max_depth": "None",
    "model__min_samples_leaf": "1",
    "model__n_estimators": "100"
  }
}

Status experimento AG 01: completed


PIPELINE DE PRE-PROCESSAMENTO
[load] Dataset carregado: 1014 linhas x 7 colunas
[outliers] Removidas 2 linha(s) com HeartRate < 30 bpm.
[age] Removidos 82 registro(s) com Age > 54 anos.
[dedup] Removidas 140 ocorrencia(s) acima do limite de 3 por grupo. Linhas restantes: 790
[features] BodyTemp convertida de Fahrenheit para Celsius.
[split] Treino: 632 linhas | Teste: 158 linhas (20% teste)
[scaler] StandardScaler aplicado (fit no treino, transform em ambos).
-------------------------------------------------------
Features finais (6): ['Age', 'SystolicBP', 'DiastolicBP', 'BS', 'BodyTemp', 'HeartRate']
Distribuicao do alvo — treino:
  low risk (0): 277 (43.8%)
  mid risk (1): 198 (31.3%)
  high risk (2): 157 (24.8%)
Distribuicao do alvo — teste:
  low risk (0): 69 (43.7%)
  mid risk (1): 50 (31.6%)
  high risk (2): 39 (24.7%)

Feature importance: {'BS': 0.3337, 'SystolicBP': 0.1891, 'Age': 0.1666, 'DiastolicBP': 0.1266, 'HeartRate': 0.1088, 'BodyTemp': 0.0752}


## 2. Pacientes de demonstração

In [3]:
RISK_DECODING = {0: "low risk", 1: "mid risk", 2: "high risk"}

patients = [
    {"Age": 25, "SystolicBP": 90, "DiastolicBP": 60, "BS": 6.0, "BodyTemp": 98.0, "HeartRate": 76, "label": "low risk"},
    {"Age": 35, "SystolicBP": 130, "DiastolicBP": 85, "BS": 8.5, "BodyTemp": 99.0, "HeartRate": 88, "label": "mid risk"},
    {"Age": 48, "SystolicBP": 160, "DiastolicBP": 100, "BS": 15.0, "BodyTemp": 101.0, "HeartRate": 105, "label": "high risk"},
]

predictions = []
for p in patients:
    features = {k: v for k, v in p.items() if k != "label"}
    df_patient = pd.DataFrame([features])
    proba = rf_pipeline.predict_proba(df_patient)[0]
    pred_idx = proba.argmax()
    pred_label = RISK_DECODING[pred_idx]
    prob_dict = {RISK_DECODING[i]: round(float(proba[i]), 4) for i in range(3)}
    predictions.append({
        "patient_data": features,
        "prediction": pred_label,
        "probabilities": prob_dict,
        "label_real": p["label"],
    })
    print(f"Paciente {p['Age']} anos | Real: {p['label']} | Predito: {pred_label} | Probs: {prob_dict}")

Paciente 25 anos | Real: low risk | Predito: high risk | Probs: {'low risk': 0.0, 'mid risk': 0.06, 'high risk': 0.94}
Paciente 35 anos | Real: mid risk | Predito: high risk | Probs: {'low risk': 0.0, 'mid risk': 0.06, 'high risk': 0.94}
Paciente 48 anos | Real: high risk | Predito: high risk | Probs: {'low risk': 0.0, 'mid risk': 0.06, 'high risk': 0.94}


## 3. Diagnóstico explicado pelo Claude

In [4]:
interpretacoes = []

for i, pred in enumerate(predictions):
    print(f"\n{'='*70}")
    print(f"PACIENTE {i+1} — {pred['label_real'].upper()} (predito: {pred['prediction'].upper()})")
    print(f"{'='*70}")

    explicacao = interpret_diagnosis(
        patient_data=pred["patient_data"],
        prediction=pred["prediction"],
        probabilities=pred["probabilities"],
        feature_importance=feature_importance,
    )
    print(explicacao)
    interpretacoes.append(explicacao)


PACIENTE 1 — LOW RISK (predito: HIGH RISK)


# Análise de Risco Gestacional — Suporte à Decisão Clínica

> ⚠️ **Aviso importante:** Esta análise é gerada por um modelo de inteligência artificial e deve ser interpretada como ferramenta de apoio — não substitui a avaliação clínica presencial nem o julgamento do médico assistente.

---

## 1. Explicação da Predição

O modelo classificou esta paciente de **25 anos como alto risco** com 94% de confiança. À primeira vista, os dados vitais isolados podem parecer dentro de faixas aceitáveis, o que torna essa predição especialmente importante de ser investigada — pois é justamente essa combinação de valores que gerou o alerta.

O ponto de maior destaque é a **glicemia** (BS = 6,0 mmol/L), que representa o fator de maior peso na decisão do modelo. Embora esse valor possa parecer limítrofe em contexto não gestacional, na gravidez os limiares diagnósticos são mais rigorosos, e esse nível — associado ao perfil pressórico desta paciente — acende um sinal de atenção relevante. Além disso, a **p

# Análise de Risco Gestacional — Suporte à Decisão Clínica

> ⚠️ **Aviso importante:** Esta análise é gerada por um modelo de inteligência artificial como ferramenta de apoio à decisão. Ela **não substitui** a avaliação clínica presencial nem o julgamento do médico assistente.

---

## 1. Explicação da Predição

O modelo classificou esta paciente como **alto risco** com 94% de confiança — uma margem muito elevada, sem sobreposição prática com as demais categorias. Isso indica que o conjunto de variáveis apresentado é consistente, dentro do padrão aprendido pelo algoritmo, com perfis de gestantes que evoluíram para desfechos de maior gravidade no dataset de treinamento.

A combinação de **glicemia elevada (BS = 8,5 mmol/L)**, **pressão arterial limítrofe-alta** (130/85 mmHg) e **idade materna avançada (35 anos)** forma um conjunto de fatores que, isoladamente, já mereceriam atenção — mas que juntos reforçam mutuamente o sinal de risco. A leve elevação da temperatura corporal (99°F / 37,

# Análise de Risco Gestacional — Suporte à Decisão Clínica

> ⚠️ **Aviso importante:** Esta análise é gerada por um modelo de inteligência artificial como ferramenta de apoio à decisão. Ela **não substitui** a avaliação clínica presencial nem o julgamento do médico assistente.

---

## 1. Explicação da Predição

O modelo classificou esta paciente como **alto risco** com 94% de confiança — uma das pontuações mais elevadas possíveis, o que indica que o conjunto de dados clínicos apresentados é fortemente sugestivo de gestação de alto risco, sem ambiguidade estatística relevante.

A combinação dos achados é particularmente preocupante: a paciente tem **48 anos**, uma faixa etária em que a gestação já carrega, por si só, risco elevado de complicações como pré-eclâmpsia, diabetes gestacional e insuficiência placentária. Associado a isso, ela apresenta **pressão arterial de 160/100 mmHg** — valores que atendem aos critérios diagnósticos de **hipertensão grave na gestação** — e **glicemia de 

## 4. Modo Q&A — Perguntas sobre o caso

In [5]:
# Paciente de alto risco (índice 2)
paciente_alto_risco = predictions[2]

perguntas = [
    "Quais são os principais sinais de alerta que devo monitorar?",
    "Esse nível de pressão arterial é preocupante para a idade da paciente?",
]

respostas_qa = []

for pergunta in perguntas:
    print(f"\n{'─'*70}")
    print(f"PERGUNTA: {pergunta}")
    print(f"{'─'*70}")

    resposta = answer_question(
        patient_data=paciente_alto_risco["patient_data"],
        prediction=paciente_alto_risco["prediction"],
        probabilities=paciente_alto_risco["probabilities"],
        question=pergunta,
    )
    print(resposta)
    respostas_qa.append({"pergunta": pergunta, "resposta": resposta})


──────────────────────────────────────────────────────────────────────
PERGUNTA: Quais são os principais sinais de alerta que devo monitorar?
──────────────────────────────────────────────────────────────────────


## Sinais de Alerta Prioritários para Monitoramento

Com base nos dados apresentados, esta paciente concentra **múltiplos fatores de risco simultâneos** que merecem atenção imediata. Veja os pontos críticos:

---

### 🔴 Atenção Imediata

| Parâmetro | Valor da Paciente | Por que preocupa |
|---|---|---|
| PA Sistólica | 160 mmHg | Limiar de hipertensão grave em gestantes |
| PA Diastólica | 100 mmHg | Critério de gravidade em pré-eclâmpsia |
| Glicemia | 15,0 mmol/L (~270 mg/dL) | Hiperglicemia importante — risco de cetoacidose |
| Temperatura | 101°F (~38,3°C) | Febre — exige investigação de foco infeccioso |
| FC | 105 bpm | Taquicardia — pode refletir febre, hipovolemia ou sepse |
| Idade | 48 anos | Risco basal elevado para todas as complicações acima |

---

### 🔍 O que monitorar ativamente

**Relacionado à hipertensão:**
- Surgimento ou piora de cefaleia, escotomas, epigastralgia
- Proteinúria (fita ou proteinúria de 24h)
- Edema de instalação rápida
- Sinais de iminência de eclâ

## Resposta sobre a Pressão Arterial desta Paciente

**Sim, esse nível pressórico é preocupante — e significativamente.**

---

### O que os dados mostram

A paciente apresenta **PA de 160/100 mmHg**, o que já ultrapassa o limiar diagnóstico de **hipertensão grave na gestação**, independentemente da idade.

> Na obstetrícia, valores ≥ 160/110 mmHg definem hipertensão grave. Esta paciente está **no limiar superior**, com diastólica já em 100 mmHg.

---

### Por que a idade agrava o quadro

Com **48 anos**, esta paciente está em faixa de **gestação de alto risco pela idade isoladamente**. Gestantes acima de 40 anos têm risco aumentado de:

- Pré-eclâmpsia e suas complicações
- Hipertensão crônica sobreposta à gestacional
- Menor reserva cardiovascular para suportar as adaptações hemodinâmicas da gravidez

A combinação **idade avançada + PA 160/100** eleva substancialmente o risco de desfechos adversos maternos e fetais.

---

### Outros dados que reforçam a preocupação

Os demais parâmet

## 5. Relatório de otimização gerado pelo Claude

In [6]:
baseline_rf = baseline_results["random_forest"]

ag_results_raw = experiment_01.get("results")
ag_config = experiment_01.get("config", {})
usando_dados_simulados = False

if ag_results_raw is None or experiment_01.get("status") == "pending":
    usando_dados_simulados = True
    print("[SIMULADO] AG ainda em execucao — usando dados de exemplo para demonstracao")
    ag_results = {
        "f1_macro": 0.915,
        "roc_auc_macro_ovr": 0.985,
        "recall_high_risk": 0.974,
        "recall_low_risk": 0.899,
        "recall_mid_risk": 0.900,
        "precision_macro": 0.912,
        "accuracy": 0.911,
    }
else:
    # Metricas ficam em results.test_metrics
    ag_results = ag_results_raw.get("test_metrics", ag_results_raw)
    print("[REAL] Usando resultados reais do Algoritmo Genetico")

baseline_para_relatorio = {k: v for k, v in baseline_rf.items() if k != "best_params"}

print(f"\nBaseline RF:  f1_macro={baseline_para_relatorio['f1_macro']} | recall_high_risk={baseline_para_relatorio['recall_high_risk']}")
print(f"Otimizado AG: f1_macro={ag_results['f1_macro']} | recall_high_risk={ag_results['recall_high_risk']}")
if usando_dados_simulados:
    print("\n[AVISO] Os resultados do AG acima sao SIMULADOS para fins de demonstracao.")

print(f"\n{'='*70}")
print("RELATORIO NARRATIVO — CLAUDE")
print(f"{'='*70}")

relatorio = generate_optimization_report(
    baseline_metrics=baseline_para_relatorio,
    optimized_metrics=ag_results,
    ga_config=ag_config,
)
print(relatorio)

[REAL] Usando resultados reais do Algoritmo Genetico

Baseline RF:  f1_macro=0.9017 | recall_high_risk=0.9487
Otimizado AG: f1_macro=0.907 | recall_high_risk=0.9744

RELATORIO NARRATIVO — CLAUDE


# Relatório de Análise — Otimização por Algoritmo Genético
### Classificador de Risco Gestacional | Random Forest

---

## 1. Análise Narrativa da Melhoria

O modelo otimizado pelo Algoritmo Genético apresentou ganhos modestos, porém clinicamente direcionados em relação ao baseline gerado por GridSearch convencional.

Em termos gerais, a acurácia global subiu de **89,9% para 90,5%**, e o F1-macro — métrica que equilibra precisão e sensibilidade entre as três classes de risco — melhorou de **0,9017 para 0,9070**. Esses números isolados poderiam ser interpretados como variação marginal. Contudo, o que torna essa otimização relevante do ponto de vista clínico **não é a melhoria média, mas sim onde ela ocorreu**.

O modelo otimizado tornou-se **significativamente mais sensível para pacientes de alto risco** (recall passou de 94,9% para **97,4%**), ao mesmo tempo em que sustentou desempenho sólido nas demais classes. A única concessão observada foi uma leve queda no recall de risco intermed

## 6. Conclusão

Este notebook demonstrou as três funcionalidades principais do módulo de integração com LLM:

1. **`interpret_diagnosis`** — Explica em linguagem natural a predição do modelo para cada paciente, destacando os fatores clínicos mais relevantes e o que o médico deve monitorar.

2. **`answer_question`** — Permite ao médico fazer perguntas livres sobre o caso, recebendo respostas contextualizadas com os dados da paciente e a predição do modelo.

3. **`generate_optimization_report`** — Gera um relatório narrativo comparando o modelo baseline com o otimizado pelo Algoritmo Genético, traduzindo as métricas em implicações clínicas.

O Claude atua como camada de **interpretabilidade clínica**, tornando as saídas do modelo de ML acessíveis e acionáveis para o médico obstetra.

In [7]:
llm_examples = {
    "interpretacoes": [
        {
            "paciente": pred["patient_data"],
            "label_real": pred["label_real"],
            "prediction": pred["prediction"],
            "probabilities": pred["probabilities"],
            "explicacao_claude": interpretacoes[i],
        }
        for i, pred in enumerate(predictions)
    ],
    "qa_alto_risco": respostas_qa,
    "relatorio_otimizacao": {
        "dados_simulados": usando_dados_simulados,
        "baseline": baseline_para_relatorio,
        "otimizado": ag_results,
        "ga_config": ag_config,
        "relatorio_claude": relatorio,
    },
}

output_path = EXPERIMENTS_DIR / "llm_examples.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(llm_examples, f, indent=2, ensure_ascii=False)

print(f"Exemplos salvos em: {output_path}")

Exemplos salvos em: /Users/igornatanael/workspace/tech-challenge-fiap-2/experiments/llm_examples.json
